# Algorithm Nesting and Hybrid Strategies

## Overview

NSGABLACK supports various hybrid strategies combining multiple algorithms for improved performance.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Callable

print("Algorithm Nesting Tutorial")

## Nested Optimization Strategy

In [ ]:
class NestedOptimizer:
    def __init__(self, outer_algorithm, inner_algorithm):
        self.outer = outer_algorithm
        self.inner = inner_algorithm
        self.best_solution = None
        self.best_fitness = float('inf')
    
    def optimize(self, problem, n_outer=20, n_inner=50):
        for outer_iter in range(n_outer):
            # Outer algorithm generates starting points
            start_points = self.outer.generate_starting_points(problem, 5)
            
            for start_point in start_points:
                # Inner algorithm refines each point
                solution, fitness = self.inner.optimize(problem, start_point, n_inner)
                
                if fitness < self.best_fitness:
                    self.best_fitness = fitness
                    self.best_solution = solution
            
            if outer_iter % 5 == 0:
                print(f"Outer iteration {outer_iter}: Best fitness = {self.best_fitness:.4f}")
        
        return self.best_solution, self.best_fitness

# Example algorithms
class RandomSearchAlgorithm:
    def generate_starting_points(self, problem, n_points):
        return [np.random.uniform(-5, 5, problem.dimension) for _ in range(n_points)]

class LocalSearchAlgorithm:
    def optimize(self, problem, start_point, n_iter):
        current = start_point.copy()
        current_fitness = problem.evaluate(current)
        
        for _ in range(n_iter):
            # Simple hill climbing
            neighbor = current + np.random.normal(0, 0.1, len(current))
            neighbor = np.clip(neighbor, -5, 5)
            neighbor_fitness = problem.evaluate(neighbor)
            
            if neighbor_fitness < current_fitness:
                current = neighbor
                current_fitness = neighbor_fitness
        
        return current, current_fitness

# Test nested optimization
class TestProblem:
    def __init__(self, dimension=5):
        self.dimension = dimension
    
    def evaluate(self, x):
        return np.sum(x**2) + 10*np.sin(np.sum(x))

problem = TestProblem(dimension=5)
nested_opt = NestedOptimizer(RandomSearchAlgorithm(), LocalSearchAlgorithm())
best_sol, best_fit = nested_opt.optimize(problem)

print(f"\nNested optimization complete!")
print(f"Best fitness: {best_fit:.4f}")

## Parallel Hybrid Strategy

In [ ]:
class ParallelHybridOptimizer:
    def __init__(self, algorithms):
        self.algorithms = algorithms
        self.results = []
    
    def optimize(self, problem, n_iter=100):
        # Run all algorithms in parallel (simplified)
        for alg_name, algorithm in self.algorithms.items():
            print(f"Running {alg_name}...")
            solution, fitness = algorithm.optimize(problem, n_iter)
            self.results.append({
                'algorithm': alg_name,
                'solution': solution,
                'fitness': fitness
            })
        
        # Select best result
        best_result = min(self.results, key=lambda x: x['fitness'])
        return best_result['solution'], best_result['fitness']

# Define different algorithms
class MonteCarloAlgorithm:
    def optimize(self, problem, n_iter):
        best = None
        best_fitness = float('inf')
        
        for _ in range(n_iter):
            x = np.random.uniform(-5, 5, problem.dimension)
            fitness = problem.evaluate(x)
            if fitness < best_fitness:
                best = x
                best_fitness = fitness
        
        return best, best_fitness

class GreedyAlgorithm:
    def optimize(self, problem, n_iter):
        current = np.random.uniform(-5, 5, problem.dimension)
        step_size = 0.5
        
        for _ in range(n_iter):
            current_fitness = problem.evaluate(current)
            improved = False
            
            for i in range(len(current)):
                for delta in [-step_size, step_size]:
                    neighbor = current.copy()
                    neighbor[i] += delta
                    neighbor = np.clip(neighbor, -5, 5)
                    neighbor_fitness = problem.evaluate(neighbor)
                    
                    if neighbor_fitness < current_fitness:
                        current = neighbor
                        current_fitness = neighbor_fitness
                        improved = True
            
            if not improved:
                step_size *= 0.9
        
        return current, current_fitness

# Test parallel hybrid
algorithms = {
    'MonteCarlo': MonteCarloAlgorithm(),
    'Greedy': GreedyAlgorithm(),
    'LocalSearch': LocalSearchAlgorithm()
}

parallel_opt = ParallelHybridOptimizer(algorithms)
best_sol, best_fit = parallel_opt.optimize(problem, n_iter=50)

print(f"\nParallel hybrid complete!")
print(f"Best fitness: {best_fit:.4f}")

# Show all results
for result in parallel_opt.results:
    print(f"{result['algorithm']}: {result['fitness']:.4f}")

## Sequential Hybrid Strategy

In [ ]:
class SequentialHybridOptimizer:
    def __init__(self, algorithm_sequence):
        self.sequence = algorithm_sequence
        self.history = []
    
    def optimize(self, problem, n_iter_per_alg=50):
        current_solution = np.random.uniform(-5, 5, problem.dimension)
        current_fitness = problem.evaluate(current_solution)
        
        for i, (alg_name, algorithm) in enumerate(self.sequence):
            print(f"\nStage {i+1}: Running {alg_name}")
            print(f"Initial fitness: {current_fitness:.4f}")
            
            # Run algorithm with current best as starting point
            new_solution, new_fitness = algorithm.optimize_from(
                problem, current_solution, n_iter_per_alg
            )
            
            if new_fitness < current_fitness:
                current_solution = new_solution
                current_fitness = new_fitness
                print(f"Improved! New fitness: {current_fitness:.4f}")
            else:
                print(f"No improvement. Fitness: {current_fitness:.4f}")
            
            self.history.append(current_fitness)
        
        return current_solution, current_fitness

# Enhance algorithms to support starting from a point
class EnhancedMonteCarlo(MonteCarloAlgorithm):
    def optimize_from(self, problem, start_point, n_iter):
        best = start_point.copy()
        best_fitness = problem.evaluate(best)
        
        for _ in range(n_iter):
            x = best + np.random.normal(0, 1, len(best))
            x = np.clip(x, -5, 5)
            fitness = problem.evaluate(x)
            if fitness < best_fitness:
                best = x
                best_fitness = fitness
        
        return best, best_fitness

# Test sequential hybrid
sequence = [
    ('MonteCarlo', EnhancedMonteCarlo()),
    ('Greedy', GreedyAlgorithm()),
    ('LocalSearch', LocalSearchAlgorithm())
]

sequential_opt = SequentialHybridOptimizer(sequence)
best_sol, best_fit = sequential_opt.optimize(problem, n_iter_per_alg=30)

print(f"\nSequential hybrid complete!")
print(f"Final best fitness: {best_fit:.4f}")

# Plot progress
plt.figure(figsize=(8, 4))
plt.plot(sequential_opt.history, 'bo-', markersize=8)
plt.xlabel('Algorithm Stage')
plt.ylabel('Best Fitness')
plt.title('Sequential Hybrid Progress')
plt.xticks(range(len(sequence)), [name for name, _ in sequence])
plt.grid(True, alpha=0.3)
plt.show()

## Cooperative Coevolution

Different algorithms work together sharing information.

In [ ]:
class CooperativeOptimizer:
    def __init__(self, algorithms, share_interval=10):
        self.algorithms = algorithms
        self.share_interval = share_interval
        self.shared_best = None
        self.shared_fitness = float('inf')
    
    def optimize(self, problem, n_iter=100):
        # Initialize each algorithm
        algorithm_states = {}
        for name, alg in self.algorithms.items():
            algorithm_states[name] = {
                'algorithm': alg,
                'current': np.random.uniform(-5, 5, problem.dimension),
                'fitness': float('inf')
            }
        
        for iteration in range(n_iter):
            # Run each algorithm for one iteration
            for name, state in algorithm_states.items():
                # Simplified single iteration
                new_solution = state['current'] + np.random.normal(0, 0.1, problem.dimension)
                new_solution = np.clip(new_solution, -5, 5)
                new_fitness = problem.evaluate(new_solution)
                
                if new_fitness < state['fitness']:
                    state['current'] = new_solution
                    state['fitness'] = new_fitness
                    
                    # Update global best
                    if new_fitness < self.shared_fitness:
                        self.shared_best = new_solution.copy()
                        self.shared_fitness = new_fitness
            
            # Share best solution periodically
            if iteration % self.share_interval == 0 and self.shared_best is not None:
                for name, state in algorithm_states.items():
                    # Inject shared solution with probability
                    if np.random.rand() < 0.3:
                        state['current'] = self.shared_best + np.random.normal(0, 0.01, problem.dimension)
            
            if iteration % 20 == 0:
                print(f"Iteration {iteration}: Shared best fitness = {self.shared_fitness:.4f}")
        
        return self.shared_best, self.shared_fitness

# Test cooperative optimization
cooperative_algorithms = {
    'Explorer': MonteCarloAlgorithm(),
    'Exploiter': GreedyAlgorithm()
}

cooperative_opt = CooperativeOptimizer(cooperative_algorithms)
best_sol, best_fit = cooperative_opt.optimize(problem, n_iter=100)

print(f"\nCooperative optimization complete!")
print(f"Best fitness: {best_fit:.4f}")

## Summary

Hybrid strategies in NSGABLACK:
- Nested: Algorithms within algorithms
- Parallel: Multiple algorithms compete
- Sequential: Algorithms run in sequence
- Cooperative: Algorithms share information

## Next Tutorial

Next: [Surrogate Models Explained](08_Surrogate_Models_Explained.ipynb)